In [ ]:
import yfinance as yf
import pandas as pd
import ta
import requests
import json
from datetime import datetime, timezone
import time

In [32]:
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "qwen3:4b"

In [ ]:
def get_session():
    hour = datetime.now(timezone.utc).hour
    
    if 7 <= hour < 16:
        return "London"
    elif 13 <= hour < 22:
        return "New York"
    else:
        return "Asian"

In [34]:
def fetch_data():
    df = yf.download("EURUSD=X", interval="5m", period="1d", auto_adjust=False)
    
    # Flatten MultiIndex columns if present
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    
    # RSI
    df["rsi"] = ta.momentum.RSIIndicator(close=df["Close"], window=14).rsi()
    
    # MACD
    macd = ta.trend.MACD(close=df["Close"])
    df["macd"] = macd.macd()
    df["macd_signal"] = macd.macd_signal()
    
    return df.dropna()

In [35]:
def build_signal(df):
    last = df.iloc[-1]
    
    return {
        "pair": "EUR/USD",
        "price": float(last["Close"]),
        "rsi": float(last["rsi"]),
        "macd": "bullish" if last["macd"] > last["macd_signal"] else "bearish",
        "trend": "uptrend" if last["Close"] > df["Close"].rolling(20).mean().iloc[-1] else "downtrend",
        "session": get_session(),
        "volatility": "high" if df["Close"].pct_change().std() > 0.0005 else "low"
    }

In [ ]:
def analyze_signal(data):
    prompt = f"""Based on this forex data, respond ONLY with valid JSON (no explanation):

{json.dumps(data, indent=2)}

Output format:
{{"signal": "BUY or SELL or HOLD", "confidence": 0-100, "reason": "brief reason"}}"""
    
    try:
        response = requests.post(
            OLLAMA_URL,
            json={
                "model": MODEL,
                "prompt": prompt,
                "stream": False,
                "format": "json",  # Force JSON output
                "options": {
                    "temperature": 0.1,
                    "num_predict": 500
                }
            },
            timeout=30
        )
        
        # Check HTTP status
        response.raise_for_status()
        
        # Parse response
        response_data = response.json()
        
        # Check both 'response' and 'thinking' fields
        raw_response = response_data.get("response", "").strip()
        if not raw_response:
            raw_response = response_data.get("thinking", "").strip()
        
        print("Raw LLM response:", raw_response)
        
        if not raw_response:
            raise ValueError(f"Empty response from LLM. Full response: {response_data}")
        
        # Try to extract JSON from the response
        try:
            # First, try parsing directly
            return json.loads(raw_response)
        except json.JSONDecodeError:
            # Try to find JSON object in the response
            import re
            json_match = re.search(r'\{[^{}]+\}', raw_response, re.DOTALL)
            if json_match:
                return json.loads(json_match.group())
            else:
                raise ValueError(f"No valid JSON found in response: {raw_response}")
    
    except requests.exceptions.ConnectionError:
        raise ConnectionError(f"Cannot connect to Ollama at {OLLAMA_URL}. Is Ollama running?")
    except requests.exceptions.Timeout:
        raise TimeoutError("Request to Ollama timed out")
    except requests.exceptions.HTTPError as e:
        raise RuntimeError(f"HTTP error from Ollama: {e}")

In [37]:
df = fetch_data()
signal = build_signal(df)

print("Signal input:")
print(signal)

result = analyze_signal(signal)

print("\nLLM output:")
print(result)

[*********************100%***********************]  1 of 1 completed
C:\Users\nboud\AppData\Local\Temp\ipykernel_27156\2796563963.py:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  hour = datetime.utcnow().hour


Signal input:
{'pair': 'EUR/USD', 'price': 1.1551345586776733, 'rsi': 31.33788090540324, 'macd': 'bearish', 'trend': 'downtrend', 'session': 'London', 'volatility': 'low'}
Full API response: {'model': 'qwen3:4b', 'created_at': '2026-03-26T08:06:04.0887134Z', 'response': '', 'thinking': 'We are given the following data for EUR/USD:\n   - price: 1.1551345586776733\n   - rsi: 31.33788090540324 (which is below 40, indicating oversold but in a downtrend)\n   - macd: bearish (so the trend is bearish)\n   - trend: downtrend (confirmed)\n   - session: London (which is typically the session when the market is active and liquidity is high, but note that the London session is when the US and European sessions overlap, so it\'s a busy session)\n   - volatility: low\n\nAs a conservative forex day trader, I must avoid risky trades. The current trend is downtrend and macd is bearish, so the market is bearish.\n\nThe RSI is 31.: This is in the oversold region (typically below 30 is considered oversold

ValueError: No valid JSON found in response: We are given the following data for EUR/USD:
   - price: 1.1551345586776733
   - rsi: 31.33788090540324 (which is below 40, indicating oversold but in a downtrend)
   - macd: bearish (so the trend is bearish)
   - trend: downtrend (confirmed)
   - session: London (which is typically the session when the market is active and liquidity is high, but note that the London session is when the US and European sessions overlap, so it's a busy session)
   - volatility: low

As a conservative forex day trader, I must avoid risky trades. The current trend is downtrend and macd is bearish, so the market is bearish.

The RSI is 31.: This is in the oversold region (typically below 30 is considered oversold). However, in a downtrend, oversold conditions might not be a strong reversal signal because the trend is still bearish.

Given the low volatility, the market is less likely to have big moves. But note: the London session is active and might have more l